# Multiple lineare Regression – numerische und kategoriale Features

**Ziel:** Eine CSV hochladen und eine **multiple lineare Regression** mit numerischen (z. B. `age`, `goalsTotal`) und kategorialen Features (z. B. `position`) trainieren.

**Ablauf:** CSV hochladen → Variablen wählen → Zusammenhänge visualisieren → Kategorien automatisch per **One-Hot-Encoding** in 0/1-Spalten umwandeln → Regression trainieren → Koeffizienten, Prognosen und Fehler auswerten.

**Beispielkonfigurationen:**
- Advertising: `NUMERIC_FEATURES = ['TV', 'radio', 'newspaper']`, `CATEGORICAL_FEATURES = []`, `TARGET = 'sales'`
- Fußballspieler: `NUMERIC_FEATURES = ['age', 'goalsTotal']`, `CATEGORICAL_FEATURES = ['position']`, `TARGET = 'marketvalue'`

**Wichtig:** Nominale Kategorien wie Position **nicht** als `1, 2, 3` mit künstlicher Reihenfolge codieren. Stattdessen erzeugen wir Dummy-Variablen und lassen eine **Referenzkategorie** weg, um Redundanz im Modell zu vermeiden. Die Umwandlung wird ausschließlich auf Trainingsdaten gelernt und danach auf Testdaten angewendet.
**Bedienung:** Führen Sie die Zellen nacheinander mit ▶ aus. Die technischen Code-Zellen sind eingeklappt; nur im Abschnitt **Konfiguration** bearbeiten Sie den Python-Code. Über „Code anzeigen“ können Sie die verborgenen Zellen bei Bedarf öffnen.


## 1. CSV-Datei hochladen

In [ ]:
# @title CSV hochladen { display-mode: "form" }
from google.colab import files
# Öffnet den Datei-Upload in Google Colab.
uploaded = files.upload()
if not uploaded:
    raise ValueError('Bitte eine CSV-Datei auswählen.')

## 2. Daten einlesen und untersuchen

In [ ]:
# @title Daten einlesen und Überblick anzeigen { display-mode: "form" }
import io
import pandas as pd
import numpy as np

filename = next(iter(uploaded))
# Liest die hochgeladene CSV-Datei als Tabelle ein.
df = pd.read_csv(io.BytesIO(uploaded[filename]))
# Entfernt automatisch gespeicherte Indexspalten.
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed(:|$)')]

print('Datensatz:', filename)
print('Zeilen, Spalten:', df.shape)
print('Spalten und Datentypen:')
display(pd.DataFrame({'Datentyp': df.dtypes.astype(str), 'Fehlende Werte': df.isna().sum()}))
print('Numerische Spalten:', df.select_dtypes(include='number').columns.tolist())
print('Kategoriale Spalten:', df.select_dtypes(exclude='number').columns.tolist())
display(df.head())
display(df.describe(include='number').T.round(2))

## 3. Konfiguration – numerische und kategoriale Features wählen

Verwenden Sie die **exakten Spaltennamen** aus der Ausgabe oben. `NUMERIC_FEATURES` enthalten Zahlen, `CATEGORICAL_FEATURES` Kategorien (z. B. `position`). Die Zielvariable muss numerisch sein.

Für Fußballspieler z. B. `['age', 'goalsTotal']` und `['position']` wählen. `name` ist normalerweise keine geeignete Eingabevariable: fast jeder Name bildet eine eigene Kategorie.

In [ ]:
# ===== KONFIGURATION: HIER ANPASSEN =====
NUMERIC_FEATURES = ['TV', 'radio', 'newspaper']
CATEGORICAL_FEATURES = []   # Fußballspieler: ['position']
TARGET = 'sales'
TEST_SIZE = 0.20
RANDOM_STATE = 42
# Beispiel Fußballspieler:
# NUMERIC_FEATURES = ['age', 'goalsTotal']
# CATEGORICAL_FEATURES = ['position']
# TARGET = 'marketvalue'
# =========================================

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
if not FEATURES:
    raise ValueError('Bitte mindestens ein Feature wählen.')
if len(set(FEATURES)) != len(FEATURES):
    raise ValueError('Features dürfen nicht doppelt vorkommen.')
if TARGET in FEATURES:
    raise ValueError('Zielvariable darf nicht als Feature ausgewählt werden.')
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f'Spalten fehlen: {missing}. Vorhanden: {df.columns.tolist()}')
non_numeric = [c for c in NUMERIC_FEATURES + [TARGET] if not pd.api.types.is_numeric_dtype(df[c])]
if non_numeric:
    raise ValueError(f'Diese Spalten müssen numerisch sein: {non_numeric}')

# Entfernt unvollständige/ungültige Zeilen der ausgewählten Variablen.
model_df = df[FEATURES + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna().copy()
# Kategorien als Strings behandeln, damit scikit-learn sie konsistent codieren kann.
for c in CATEGORICAL_FEATURES:
    model_df[c] = model_df[c].astype(str)
if len(model_df) < 10:
    raise ValueError('Zu wenige vollständige Zeilen (mindestens 10).')

print('Numerische Features:', NUMERIC_FEATURES)
print('Kategoriale Features:', CATEGORICAL_FEATURES)
print('Zielvariable:', TARGET)
print('Verwendbare Beobachtungen:', len(model_df), '| Unvollständige Zeilen entfernt:', len(df)-len(model_df))
for c in CATEGORICAL_FEATURES:
    print(f'Kategorien für {c}:', model_df[c].value_counts().to_dict())

## 4. Explorative Analyse

- **Numerische Features:** Scatterplot gegen die Zielvariable.
- **Kategoriale Features:** Boxplot der Zielvariable für jede Kategorie (bei sehr vielen Kategorien nur die zehn häufigsten).

Ein Scatterplot zeigt nur einen **bivariaten** Zusammenhang. Das spätere Regressionsmodell schätzt die Effekte aller Features gemeinsam.

In [ ]:
# @title Daten visualisieren { display-mode: "form" }
import matplotlib.pyplot as plt

if NUMERIC_FEATURES:
    # Zeigt lineare Korrelationen zwischen numerischen Variablen.
    display(model_df[NUMERIC_FEATURES + [TARGET]].corr(numeric_only=True).round(3))
    fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(5 * len(NUMERIC_FEATURES), 4), squeeze=False)
    for ax, feature in zip(axes.ravel(), NUMERIC_FEATURES):
        # Scatterplot: Zusammenhang zwischen Feature und Zielvariable.
        ax.scatter(model_df[feature], model_df[TARGET], alpha=0.45, s=14)
        ax.set(xlabel=feature, ylabel=TARGET, title=f'{TARGET} vs. {feature}')
        ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

for feature in CATEGORICAL_FEATURES:
    categories = model_df[feature].value_counts().head(10).index.tolist()
    data = [model_df.loc[model_df[feature] == cat, TARGET].to_numpy() for cat in categories]
    plt.figure(figsize=(max(6, 1.1 * len(categories)), 4))
    # Boxplot: Zielwertverteilung für verschiedene Kategorien.
    plt.boxplot(data, tick_labels=categories, showfliers=False)
    plt.title(f'{TARGET} nach {feature} (max. 10 häufigste Kategorien)')
    plt.ylabel(TARGET)
    plt.xticks(rotation=35, ha='right')
    plt.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    plt.show()

## 5. Trainings- und Testdaten aufteilen

80 % werden standardmäßig zum Training genutzt, 20 % zur Bewertung auf bislang ungesehenen Daten.

In [ ]:
# @title Trainings- und Testdaten aufteilen { display-mode: "form" }
from sklearn.model_selection import train_test_split
X = model_df[FEATURES]
y = model_df[TARGET]
# Teilt Daten in Training (Modelllernen) und Test (unabhängige Bewertung).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print('Training:', len(X_train), '| Test:', len(X_test))

## 6. One-Hot-Encoding und multiple lineare Regression

Das Notebook codiert kategoriale Variablen mittels **One-Hot-Encoding** (Dummy-Variablen). Eine Kategorie wird als **Referenzkategorie** weggelassen (`drop='first'`).

Beispiel `position`: Wenn `Defender` die Referenz ist, stehen `position_Forward` und `position_Midfielder` als 0/1-Features im Modell. Ihre Koeffizienten zeigen den Unterschied zur Referenzkategorie – **bei gleichen Werten der übrigen Features**.

Die Transformation wird nur an den Trainingsdaten angepasst. Das verhindert Datenleckage.

In [ ]:
# @title Kategorien codieren und Modell trainieren { display-mode: "form" }
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

transformers = []
if NUMERIC_FEATURES:
    transformers.append(('num', 'passthrough', NUMERIC_FEATURES))
if CATEGORICAL_FEATURES:
    # One-Hot-Encoding: Kategorien in 0/1-Spalten umwandeln; erste Kategorie ist Referenz.
    transformers.append(('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES))

# ColumnTransformer verarbeitet numerische und kategoriale Features getrennt.
preprocess = ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)
# Pipeline verknüpft Datenvorbereitung und lineare Regression.
model = Pipeline([('preprocess', preprocess), ('regression', LinearRegression())])
# fit(): Lernt die Codierung und die Regressionskoeffizienten aus Trainingsdaten.
model.fit(X_train, y_train)

term_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['regression'].coef_
intercept = model.named_steps['regression'].intercept_
print('Neue numerische Spalten nach Codierung:', list(term_names))
print(f'Intercept: {intercept:.5f}')
display(pd.DataFrame({'Term': term_names, 'Koeffizient': coefficients}).round(5))

if CATEGORICAL_FEATURES:
    encoder = model.named_steps['preprocess'].named_transformers_['cat']
    print('Referenzkategorien (jeweils weggelassen):')
    for feature, categories in zip(CATEGORICAL_FEATURES, encoder.categories_):
        print(f'  {feature}: {categories[0]} (Referenz); weitere: {list(categories[1:])}')

print('Geschätzte Regressionsfunktion:')
terms = [f'({coef:+.5f} × {name})' for name, coef in zip(term_names, coefficients)]
print(f'{TARGET}_vorhergesagt = {intercept:.5f} ' + ' '.join(terms))
print('Bei Dummy-Variablen gilt: 1 = Kategorie vorhanden, 0 = sonst.')

## 7. Vorhersagen für Testdaten und Modellfehler

**SSE / RSS:** Summe der quadrierten Fehler; **MSE:** mittlerer quadrierter Fehler; **RMSE:** Wurzel des MSE; **MAE:** durchschnittlicher absoluter Fehler; **R²:** Güte relativ zur Mittelwertprognose.

In [ ]:
# @title Vorhersagefehler berechnen { display-mode: "form" }
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# predict(): Berechnet Vorhersagen für bisher ungesehene Testdaten.
y_pred = model.predict(X_test)
resultate = X_test.copy()
resultate['Istwert'] = y_test.to_numpy()
resultate['Prognose'] = y_pred
resultate['Residuum'] = resultate['Istwert'] - resultate['Prognose']
display(resultate.head(10).round(3))

# MSE: Durchschnitt der quadrierten Prognosefehler.
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
sse = float(np.sum((y_test.to_numpy() - y_pred)**2))
rmse = np.sqrt(mse)
# R²: Modellgüte im Vergleich zur Vorhersage mit dem Testdaten-Mittelwert.
r2 = r2_score(y_test, y_pred)
print(f'R²       = {r2:.3f}')
print(f'MAE      = {mae:.3f}')
print(f'MSE      = {mse:.3f}')
print(f'RMSE     = {rmse:.3f}')
print(f'SSE/RSS  = {sse:.3f}')
print('Alle Kennzahlen wurden auf dem TEST-Datensatz berechnet.')

## 8. Tatsächliche und vorhergesagte Zielwerte vergleichen

In [ ]:
# @title Vorhersagen visualisieren { display-mode: "form" }
plt.figure(figsize=(6,5))
# Vergleicht tatsächliche Werte mit den Modellvorhersagen.
plt.scatter(y_test, y_pred, alpha=0.75)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
plt.plot([lo,hi], [lo,hi], linestyle='--', color='black', label='Perfekte Prognose')
plt.xlabel(f'{TARGET} – tatsächlich')
plt.ylabel(f'{TARGET} – vorhergesagt')
plt.title('Tatsächlich vs. vorhergesagt')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 9. Reflexion

1. Welche Features sind numerisch, welche kategorial? Welche Kategorien kommen bei `position` vor?
2. Wie viele zusätzliche 0/1-Spalten wurden erzeugt? Welche Kategorie ist die **Referenz**?
3. Was sagt beispielsweise der Koeffizient für `position_Forward` **gegenüber der Referenzposition** aus, wenn Alter und Tore gleich bleiben?
4. Warum wäre es problematisch, `Forward = 1`, `Midfielder = 2`, `Defender = 3` als normale Zahl in die Regression zu geben?
5. Wie verändern sich RMSE und R², wenn `position` zusätzlich zu `age` und `goalsTotal` berücksichtigt wird?

**Wichtig:** Die Regression unterstellt weiterhin lineare/additive Effekte in den verwendeten Termen. Die Dummy-Codierung schafft keinen numerischen Abstand zwischen Positionen. Korrelations- oder Regressionszusammenhänge beweisen keine Kausalität.